# LeNet-5 Neural Network for MNIST


### Options


In [68]:
batch_size = 20
training_epochs = batch_size * 40

### Load & convert data


In [41]:
import tensorflow as tf
from tensorflow import keras
import numpy as np


def convert_data(x_data: np.ndarray, y_data: np.ndarray):
    # hacky way to increase the size of a (unknown)x28x28 to (unknown)x32x32
    # by appending two zeros at all sides of the 28x28 matrix
    tmp = np.zeros((x_data.shape[0], 2, x_data.shape[2]))
    x_data = np.append(x_data, tmp, axis=1)
    x_data = np.append(tmp, x_data, axis=1)
    tmp = np.zeros((x_data.shape[0], x_data.shape[1], 2))
    x_data = np.append(x_data, tmp, axis=2)
    x_data = np.append(tmp, x_data, axis=2)

    # change dimensions: (unknown)x32x32 to (unknown)x32x32x1
    x_data = np.expand_dims(x_data, 1)
    # change dimensions: (unknown) to (unknown)x10
    y_data = keras.utils.to_categorical(y_data, 10)

    # return (x_data - 128, y_data)
    return (x_data / 256.0, y_data)


# for now we are just interested in the training data and not the test data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

(X_train, y_train) = convert_data(X_train, y_train)
(X_test, y_test) = convert_data(X_test, y_test)

In [42]:
# save data
np.save("export/mnist/x_train.npy", X_train)
np.save("export/mnist/y_train.npy", y_train)
np.save("export/mnist/x_test.npy", X_test)
np.save("export/mnist/y_test.npy", y_test)

In [43]:
print("Shapes:")
print((X_train.shape, y_train.shape))

Shapes:
((60000, 1, 32, 32), (60000, 10))


### Setup our model


In [ ]:
from torch import nn
import torch

import brevitas.nn as qnn
from brevitas.core.quant import QuantType
import torch.nn.functional as F


class QuantLeNet(nn.Module):
    def __init__(self):
        super(QuantLeNet, self).__init__()
        self.conv1 = qnn.QuantConv2d(
            1, 6, 5, weight_quant_type=QuantType.INT, weight_bit_width=8
        )
        self.relu1 = qnn.QuantReLU(quant_type=QuantType.INT, bit_width=8, max_val=6)
        self.conv2 = qnn.QuantConv2d(
            6, 16, 5, weight_quant_type=QuantType.INT, weight_bit_width=8
        )
        self.relu2 = qnn.QuantReLU(quant_type=QuantType.INT, bit_width=8, max_val=6)
        self.fc1 = qnn.QuantLinear(
            16 * 5 * 5,
            120,
            bias=True,
            weight_quant_type=QuantType.INT,
            weight_bit_width=8,
        )
        # self.relu3 = qnn.QuantReLU(quant_type=QuantType.INT, bit_width=8, max_val=6)
        self.fc2 = qnn.QuantLinear(
            120, 84, bias=True, weight_quant_type=QuantType.INT, weight_bit_width=8
        )
        # self.relu4 = qnn.QuantReLU(quant_type=QuantType.INT, bit_width=8, max_val=6)
        self.fc3 = qnn.QuantLinear(
            84, 10, bias=False, weight_quant_type=QuantType.INT, weight_bit_width=8
        )

    def forward(self, x):
        out = self.relu1(self.conv1(x))
        out = F.max_pool2d(out, 2)
        out = self.relu2(self.conv2(out))
        out = F.max_pool2d(out, 2)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        out = self.fc3(out)
        return out#.type(torch.LongTensor)

net = QuantLeNet()
print(net)

QuantLeNet(
  (conv1): QuantConv2d(
    1, 6, kernel_size=(5, 5), stride=(1, 1)
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
          (input_view_impl): Identity()
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverTensorView()
            )
            (stats): _Stats(
              (stats_impl): AbsMax()
            )
          )
        

In [ ]:
import torch

use_gpu = torch.cuda.is_available()
if use_gpu:
    net = net.cuda()
    print("USE GPU")
else:
    print("USE CPU")

USE CPU


### Fitting the model


In [ ]:
from torch.autograd import Variable

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.001)

nb_index = 0
for epoch in range(training_epochs):
	if nb_index + batch_size >= len(X_train):
		nb_index = 0
	else:
		nb_index = nb_index + batch_size

	mini_data = Variable(torch.from_numpy(X_train[nb_index:(nb_index + batch_size)].copy()))
	mini_label = Variable(torch.from_numpy(y_train[nb_index:(nb_index + batch_size)].copy()), requires_grad=False)
	mini_data = mini_data.type(torch.FloatTensor)
	mini_label = mini_label.type(torch.FloatTensor)
	
	if use_gpu:
		mini_data = mini_data.cuda()
		mini_label = mini_label.cuda()
	
	optimizer.zero_grad()
	mini_out = net(mini_data)
#	mini_out = mini_out.type(torch.LongTensor)
	print(mini_out)
	m = mini_out.argmax(1)
	tmp = torch.zeros(mini_out.size(), requires_grad=False).scatter(1, m.unsqueeze(1), 1)
	#tmp = F.one_hot(m, num_classes=5)

	mini_label = mini_label

	mini_loss = criterion(tmp, mini_label)
	mini_loss.backward()
	optimizer.step()

	print("Epoch = %d, Loss = %f" % (epoch + 1, mini_loss.data))


tensor([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])
tensor([[0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0]])


RuntimeError: 0D or 1D target tensor expected, multi-target not supported

## How good is our model for non-train data?


In [ ]:
# not tested yet (code taken from example)

Y_data = test.reshape(test.shape[0], 1, 28, 28)
Y_data = Y_data.astype(float)
Y_data /= 255.0
Y_data = torch.from_numpy(Y_data);
print(Y_data.size())
nb_test = test.shape[0]

net.eval()

final_prediction = np.ndarray(shape=(nb_test, 2), dtype=int)
#for each_sample in range(nb_test):
for each_sample in range(100):
	sample_data = Variable(Y_data[each_sample:each_sample + 1].clone())
	sample_data = sample_data.type(torch.FloatTensor)
	if use_gpu:
		sample_data = sample_data.cuda()
	sample_out = net(sample_data)
	_, pred = torch.max(sample_out, 1)
	final_prediction[each_sample][0] = 1 + each_sample
	final_prediction[each_sample][1] = pred.data[0]
	if (each_sample + 1) % 2000 == 0:
		print("Total tested = %d" % (each_sample + 1))



In [ ]:
net.eval()

# Input to the model
batch_size = 1

if use_gpu:
	x = torch.randn(batch_size, 1, 28, 28, requires_grad=True).cuda()
else:
	x = torch.randn(batch_size, 1, 28, 28, requires_grad=True)

torch_out = net(x)

In [ ]:
import brevitas.onnx as bo

exported_model = bo.export_qonnx(net, args=x, export_path="Lenet_qonnx.onnx", opset_version=13)